# Module 4 — The Python Ecosystem for Neuroscience

This notebook-style script is intended as a closing overview of commonly used
Python libraries in neuroscience.

Learning goals:
- get a broad overview of the neuroscience Python ecosystem
- understand which libraries are commonly used for which modalities
- see short code examples and entry points for further learning

This module is mostly descriptive. Some example imports may require additional
installation depending on the environment.

Libraries covered:
- MNE-Python
- SpikeInterface
- Neo
- Elephant
- nibabel
- Nilearn
- BrainGlobe Atlas API



In [ ]:
# %%
# # Uncomment to install packages

# !pip install mne
# !pip install spikeinterface
# !pip install probeinterface
# !pip install neo
# !pip install elephant
# !pip install quantities
# !pip install nibabel
# !pip install nilearn
# !pip install brainglobe-atlasapi

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. nibabel

Links:
- Documentation: https://nipy.org/nibabel/
- GitHub: https://github.com/nipy/nibabel

Nibabel is a low-level neuroimaging library.

Core idea:
Read and write neuroimaging file formats.

Common file formats:
- NIfTI
- Analyze
- MINC
- FreeSurfer formats

Short feature list:
- Read MRI files
- Write MRI files
- Access voxel data
- Work with affine transforms
- Handle neuroimaging metadata

In [ ]:
import nibabel as nib

# Create a synthetic 4D fMRI-like image
data = np.random.rand(16, 16, 8, 20)

affine = np.eye(4)

fmri_img = nib.Nifti1Image(data, affine)

print(fmri_img)

# Access image data
image_data = fmri_img.get_fdata()

print(image_data.shape)

## 2. Nilearn

Links:
- Documentation: https://nilearn.github.io/
- GitHub: https://github.com/nilearn/nilearn

Nilearn provides higher-level neuroimaging analysis tools.

This notebook focuses on:
- Image processing
- Connectivity analysis

Short feature list:
- Image masking
- Spatial smoothing
- Signal extraction
- Functional connectivity
- Decoding
- GLM analysis

In [ ]:
from nilearn import datasets
from nilearn import connectome
from nilearn.maskers import NiftiLabelsMasker

# Load a small atlas
schaefer2018_atlas = datasets.fetch_atlas_schaefer_2018(
    n_rois=100
)

print(schaefer2018_atlas.keys())

In [ ]:
# Load a small resting-state dataset
development_dataset = datasets.fetch_development_fmri(
    n_subjects=1
)

fmri_filename = development_dataset.func[0]

print(fmri_filename)

In [ ]:
# Extract regional time series. The masker converts voxel-level data into region-level signals.
masker = NiftiLabelsMasker(
    labels_img=schaefer2018_atlas.maps,
    standardize=True
)

time_series = masker.fit_transform(fmri_filename)

print(time_series.shape)

In [ ]:
# Compute a connectivity matrix. Here we compute simple correlation-based functional connectivity.
correlation_measure = connectome.ConnectivityMeasure(
    kind="correlation"
)

correlation_matrix = correlation_measure.fit_transform(
    [time_series]
)[0]

print(correlation_matrix.shape)

In [ ]:
# Visualize the connectivity matrix
plt.figure(figsize=(8, 8))

plt.imshow(correlation_matrix)

plt.colorbar()

plt.title("Functional connectivity matrix")

plt.show()

## 3. MNE-Python

Links:
- Documentation: https://mne.tools/stable/index.html
- GitHub: https://github.com/mne-tools/mne-python

MNE-Python is a major library for EEG and MEG analysis.

Typical workflow:
1. Load recording
2. Filter signals
3. Detect events
4. Create epochs
5. Average responses
6. Perform statistics or decoding

Core objects:
- `Raw`
- `Epochs`
- `Evoked`

Short feature list:
- EEG/MEG file reading
- Filtering
- Event handling
- Epoching
- Time-frequency analysis
- Source localization
- Machine learning support

In [ ]:
# Load a small example dataset
# MNE provides example datasets that are useful for teaching.
import mne

sample_data_path = mne.datasets.sample.data_path()

raw_path = (
    sample_data_path.as_posix()
    + "/MEG/sample/sample_audvis_filt-0-40_raw.fif"
)

raw = mne.io.read_raw_fif(raw_path, preload=False)

print(raw)

In [ ]:
# Inspect channels
print(raw.ch_names[:10])

In [ ]:
# Plot a short segment. This shows a small time window from the continuous recording.

raw.plot(duration=5, n_channels=20)

In [ ]:
# Basic filtering. Filtering is commonly used to isolate frequency ranges of interest.

# This may not run on jupyter notebooks

# filtered_raw = raw.copy().filter(l_freq=1, h_freq=40)
# print(filtered_raw)

## 4. SpikeInterface

Links:
- Documentation: https://spikeinterface.readthedocs.io/
- GitHub: https://github.com/SpikeInterface/spikeinterface

SpikeInterface supports extracellular electrophysiology workflows.

Typical workflow:
1. Load recording
2. Preprocess
3. Spike sorting
4. Quality control
5. Visualization
6. Export results

Core idea:
Provide a unified interface to many spike sorting tools.

Short feature list:
- Recording loading
- Preprocessing
- Spike sorting
- Quality metrics
- Visualization
- Reproducible workflows

In [ ]:
# %%
import spikeinterface.core as si
import probeinterface

# Simulated extracellular recording. Here we create a tiny synthetic recording with one spike-like event.
sampling_frequency = 30_000

duration_seconds = 2

n_channels = 4

n_samples = sampling_frequency * duration_seconds

rng = np.random.default_rng(42)

recording_data = rng.normal(
    0,
    20,
    size=(n_samples, n_channels)
)

spike_time = 20_000

recording_data[spike_time:spike_time + 40, 0] -= (
    np.hanning(40) * 150
)

print(recording_data.shape)

In [ ]:
# Plot a spike-like event
plt.figure(figsize=(10, 4))
plt.plot(recording_data[19_800:20_300, 0])
plt.xlabel("Sample index")
plt.ylabel("Signal")
plt.title("Synthetic extracellular spike")
plt.show()

In [ ]:
# Create a SpikeInterface recording object
recording = si.NumpyRecording(
    traces_list=[recording_data],
    sampling_frequency=sampling_frequency
)

print(recording)

### ProbeInterface

ProbeInterface is commonly used together with SpikeInterface.

Purpose:
Describe electrode probe geometry.

Examples:
- Neuropixels probes
- Tetrodes
- Custom probes

In [ ]:
probe = probeinterface.generate_linear_probe(
    num_elec=4,
    ypitch=20
)
probe

## 5. Neo

Links:
- Documentation: https://neo.readthedocs.io/
- GitHub: https://github.com/NeuralEnsemble/python-neo

Neo provides common electrophysiology data structures.

Core idea:
Standardized representations of neural recordings.

Typical objects:
- SpikeTrain
- AnalogSignal
- Segment
- Block

Short feature list:
- Standardized objects
- Many supported file formats
- Hierarchical experiment structure
- Works with Elephant

In [ ]:
import neo
import quantities as pq

# Create a spike train
spike_times = [0.1, 0.3, 0.35, 0.8, 1.2] * pq.s

spiketrain = neo.SpikeTrain(
    spike_times,
    t_stop=2.0 * pq.s
)

print(spiketrain)

In [ ]:
# Visualize spike times
plt.figure(figsize=(8, 2))
plt.eventplot(spiketrain.rescale("s").magnitude)
plt.xlabel("Time (s)")
plt.yticks([])
plt.title("Spike train")
plt.show()

## 4. Elephant

Links:
- Documentation: https://elephant.readthedocs.io/
- GitHub: https://github.com/NeuralEnsemble/elephant

Elephant builds on Neo and provides analysis methods for neural data.

Typical workflow:
1. Load Neo objects
2. Compute statistics
3. Analyze spike trains
4. Analyze synchrony/connectivity

Short feature list:
- Firing rates
- Spike histograms
- Correlations
- Synchrony analysis
- Spectral analysis
- Statistical tools

In [ ]:
import elephant
from elephant.statistics import time_histogram

# Compute a firing rate histogram
histogram = time_histogram(
    [spiketrain],
    bin_size=0.2 * pq.s
)
print(histogram)

In [ ]:
# Plot the histogram
hist_values = histogram.magnitude.flatten()
bin_centers = np.arange(len(hist_values)) * 0.2
plt.figure(figsize=(8, 4))
plt.bar(bin_centers, hist_values, width=0.18)
plt.xlabel("Time (s)")
plt.ylabel("Spike count")
plt.title("Spike histogram")
plt.show()

In [ ]:
# Plot the histogram
hist_values = histogram.magnitude.flatten()

bin_centers = np.arange(len(hist_values)) * 0.2

plt.figure(figsize=(8, 4))

plt.bar(bin_centers, hist_values, width=0.18)

plt.xlabel("Time (s)")
plt.ylabel("Spike count")

plt.title("Spike histogram")

plt.show()

## 7. BrainGlobe Atlas API

Links:
- Documentation: https://brainglobe.info/documentation/brainglobe-atlasapi/
- GitHub: https://github.com/brainglobe/brainglobe-atlasapi

BrainGlobe Atlas API provides access to brain atlases across species.

Short feature list:
- Multi-species support
- Atlas metadata
- Brain structures
- Coordinates
- Annotation volumes
- Integration with other tools

Examples of supported species:
- Mouse
- Zebrafish
- Rat
- Human

In [ ]:
from brainglobe import BrainGlobeAtlas
from brainglobe.list_atlases import get_all_atlases

# List available atlases
print(get_all_atlases())

In [ ]:
# download allen mouse atlas in 25 micron resolution
atlas = BrainGlobeAtlas("allen_mouse_25um")
print(atlas)

In [ ]:
# Access structure information
print(atlas.structures["CTX"])

## Additional neuroscience libraries

Some additional libraries commonly used in neuroscience:

| Library | Main purpose |
|---|---|
| PyNWB | standardized neurophysiology storage |
| MNE-BIDS | BIDS support for EEG/MEG |
| PyBIDS | querying BIDS datasets |
| DIPY | diffusion MRI |
| CaImAn | calcium imaging |
| BrainIAK | advanced fMRI analysis |